# Pretraining

This notebook uses the bundled Digits JSONL buffer as a self-supervised reconstruction task. There is no supervised label. Instead, the model masks pixel intensity values and learns to reconstruct them from the surrounding pixel context.


Only local dependencies are used. The digit records are already buffered as nested JSONL, so the notebook does not download data or reshape arrays at runtime.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger

import json2vec as j2v

logger.remove()

Each image is represented as a nested `pixels` array. Every pixel is a small JSON object with `row`, `column`, and `intensity`, which is closer to the structured data JSON2Vec is meant to read.


In [2]:
records = pl.read_ndjson("docs/data/digits.jsonl").head(24)

records.head()

pixels,digit
list[struct[3]],str
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]","""0"""
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]","""1"""
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]","""2"""
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]","""3"""
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]","""4"""


The `pixels` array creates a local context encoder. Row and column are categorical position hints with light masking, and intensity is numeric content with `p_mask=0.50`, so random intensities are hidden and reconstructed during training.

In [3]:
model = j2v.Model.from_schema(
    j2v.Array(
        j2v.Category("row", query="[*].pixels[*].row", max_vocab_size=8),
        j2v.Category("column", query="[*].pixels[*].column", max_vocab_size=8),
        j2v.Number("intensity", query="[*].pixels[*].intensity"),
        name="pixels",
        max_length=64,
    ),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

model.update(j2v.where("name") == "intensity", p_mask=0.50)
model.update(j2v.where("type") == "category", p_mask=0.05)

The data module streams the nested records through the model schema. No target field is required because masking supplies the training signal.

In [4]:
datamodule = j2v.PolarsDataModule(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=16,
    chunk_batch_size=32,
    sample_rate=1.0,
)

A single epoch is enough for the documentation example. The important behavior is that the same training loop works whether the task is supervised or mask-based pretraining.

In [5]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.


`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=1` reached.


The plot shows a root record encoder, a nested pixel encoder, and the masked numeric intensity field that drives the pretraining loss.

In [6]:
model.plot()

Schema
record [array]
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=8  parameters=27,765  arrays=2  
fields=3  targets=0  embeds=0  n_linear=1
└── pixels [array] 6,608 params
    record/pixels
    attention=mha  max_length=64  n_outputs=1  n_layers=1  n_heads=4  n_linear=1
    ├── row [category] 4,799 params
    │   record/pixels/row
    │   query=[*].pixels[*].row  pooling=query  max_vocab_size=8  topk=[]  weight=1.0  n_heads=4  p_mask=0.05  
    │   n_linear=1  n_bands=8  p_unavailable=0.01
    ├── column [category] 4,799 params
    │   record/pixels/column
    │   query=[*].pixels[*].column  pooling=query  max_vocab_size=8  topk=[]  weight=1.0  n_heads=4  p_mask=0.05  
    │   n_linear=1  n_bands=8  p_unavailable=0.01
    └── intensity [number] 4,951 params
        record/pixels/intensity
        query=[*].pixels[*].intensity  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.5  n_linear=1  
        jitter=0.0  n_bands=8  offset=4